# Анализ отчёта о расходах

В этой записной книжке показано, как создавать агентов, использующих плагины для обработки командировочных расходов с локальных изображений квитанций, создания электронного письма для отчёта о расходах и визуализации данных о расходах с помощью круговой диаграммы. Агенты динамически выбирают функции на основе контекста задачи.

Шаги:
1. Агент OCR обрабатывает локальное изображение квитанции и извлекает данные о командировочных расходах.
2. Агент по электронной почте создаёт письмо с отчётом о расходах.

### Пример сценария командировочных расходов:
Представьте, что вы сотрудник, который путешествует для деловой встречи в другом городе. Ваша компания возмещает все разумные расходы, связанные с поездкой. Вот разбор возможных командировочных расходов:
- Транспорт:
Авиабилеты туда и обратно из вашего родного города в город назначения.
Такси или сервисы поездок до и от аэропорта.
Местный транспорт в городе назначения (например, общественный транспорт, арендованные машины или такси).

- Проживание:
Проживание в отеле средней категории в деловом районе рядом с местом встречи на три ночи.

- Питание:
Суточное пособие на питание (завтрак, обед и ужин) в соответствии с политикой компании.

- Прочие расходы:
Плата за парковку в аэропорту.
Оплата доступа в интернет в отеле.
Чаевые или небольшие сервисные сборы.

- Документация:
Вы предоставляете все квитанции (за перелёты, такси, отель, питание и т. д.) и заполненный отчёт о расходах для получения возмещения.


## Импорт необходимых библиотек

Импортируйте необходимые библиотеки и модули для блокнота.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Определение моделей расходов

 Создайте модель Pydantic для отдельных расходов и класс ExpenseFormatter для преобразования пользовательского запроса в структурированные данные о расходах.

 Каждый расход будет представлен в формате:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Определение инструментов - Генерация электронной почты

Создайте функцию-инструмент для генерации электронной почты для подачи заявки на возмещение расходов.
- Этот инструмент использует декоратор `@tool` из Microsoft Agent Framework.
- Он вычисляет общую сумму расходов и форматирует детали в тело письма.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Инструмент для извлечения расходов на поездки из фотографий квитанций

Создайте функцию инструмента для извлечения расходов на поездки из фотографий квитанций.
- Этот инструмент использует декоратор `@tool` из Microsoft Agent Framework.
- Он считывает изображение квитанции, кодирует его в base64 и возвращает data URI для анализа агентом.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Обработка расходов

Определите агентов и свяжите их в последовательный рабочий процесс с помощью `WorkflowBuilder`.
- Агент OCR извлекает структурированные данные о расходах из изображения чека с помощью инструмента `load_receipt_image`.
- Агент Email принимает извлечённые данные и формирует профессиональное письмо с заявлением о расходах с помощью инструмента `generate_expense_email`.
- `WorkflowBuilder` с помощью `add_edge` создаёт последовательный конвейер: агент OCR → агент Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Основная функция

Создайте последовательный рабочий процесс и запустите его для обработки изображения чека и создания электронного письма с заявкой на возмещение расходов.


> **Примечание:** Этот рабочий процесс в настоящее время передаёт изображение чека в виде текста base64, который большинство чат-моделей (включая gpt-4.1-mini) не воспримут как изображение.
> Также это может превысить окно контекста модели. Предпочтительно запускать OCR с помощью Azure AI Vision (или другого инструмента OCR) и передавать только извлечённый текст, либо изменить подход, чтобы отправить изображение как сообщение `image_url`.
> Если вы хотите лишь избежать ошибок контекста, попробуйте уменьшить изображение чека или использовать модель с большим окном контекста.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:
Этот документ был переведен с использованием сервиса машинного перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия по обеспечению точности, имейте в виду, что автоматический перевод может содержать ошибки или неточности. Оригинальный документ на его исходном языке следует считать авторитетным источником. Для получения критически важной информации рекомендуется обратиться к профессиональному человеческому переводу. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования этого перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
